# Deploy and Manage MCP Servers and Custom Endpoints with the Python SDK

## Why This Matters
- Standardizes MCP server lifecycle and endpoint-management workflows across deployment, validation, access control, and cleanup.
- Keeps service URLs, tokens, and database credentials out of notebook cells through interactive prompts.
- Uses direct SDK calls and clear step-by-step cells so the workflow stays easy to follow.


## What You Will Accomplish
- Configure MCP management and endpoint-management SDK clients through prompted input.
- Deploy and inspect MCP servers, then manage custom endpoint mappings for multiple users.
- Exercise upgrade, lifecycle, and cleanup workflows through isolated notebook steps.


---

## Table of Contents

### [1. Setup & Prerequisites](#setup)
- [1.1 Imports](#imports)
- [1.2 Environment Configuration](#environment-configuration)
- [1.3 Authentication](#authentication)
- [1.4 Client Initialization](#client-initialization)
- [1.5 Reusable Helpers](#reusable-helpers)

### [2. MCP Management APIs](#mcp-management)
- [2.1 Run Management Health Checks and List Current Deployments](#)
- [2.2 Deploy the Sales MCP Server](#)
- [2.3 Check Deployment Status](#)

### [3. Endpoint Management APIs](#endpoint-management)
- [3.1 Create Database Users and Grant Roles Safely](#endpoint-users)
- [3.2 Initialize Per-User Endpoint Management Clients](#endpoint-clients)
- [3.3 Run Endpoint Health, Version, and Visibility Checks](#endpoint-health-version-visibility)
- [3.4 Create Endpoint Mappings](#endpoint-create-mappings)
- [3.5 Update and Delete Endpoint Mappings](#endpoint-update-delete-mappings)
  - [3.5.1 Update Endpoint Mappings](#endpoint-update-mappings)
  - [3.5.2 Delete Endpoint Mappings](#endpoint-delete-mappings)

### [4. Additional MCP Server Operations](#additional-operations)
- [4.1 Deploy the Finance MCP Server](#)
- [4.2 Upgrade Server Resources](#)
- [4.3 Managed Server Lifecycle Operations](#managed-server-lifecycle)
  - [4.3.1 Stop Managed Servers](#stop-managed-servers)
  - [4.3.2 Start Managed Servers](#start-managed-servers)
  - [4.3.3 Restart Managed Servers](#restart-managed-servers)

### [5. Cleanup and Results](#cleanup)
- [5.1 Undeploy Servers and Verify Cleanup](#)
- [5.2 Results and Interpretation](#)
- [5.3 Summary / Next Steps](#)
---


<a id="setup"></a>
## 1. Setup & Prerequisites

Run these cells in order to load imports, capture environment-specific settings, create the bearer-auth object, initialize SDK clients, and define the reusable output helper used throughout the notebook.


<a id="imports"></a>
### 1.1 Imports

This step imports the SDK modules and supporting packages used by the notebook.

In [ ]:
# Import secure prompt handling for credentials and environment-specific values.
from getpass import getpass

# Import pprint for the reusable notebook output helper defined later.
from pprint import pprint

# Import the Teradata SQL driver used for user and role setup.
import teradatasql

# Import SDK authentication helpers, service clients, and request models.
from teradata_agentstack._auth_modes import BearerAuth, BasicAuth
from teradata_agentstack.mcp_management import MCPManagementClient, Servers, Health as MgmtHealth
from teradata_agentstack.mcp_management.models import DeployRequest, UpgradeResourcesRequest
from teradata_agentstack.mcp_endpoint_management import MCPEndpointManagementClient, Endpoints, Health as EpHealth
from teradata_agentstack.mcp_endpoint_management.models import EndpointCreateRequest, EndpointUpdateRequest, EndpointDeleteRequest


<a id="environment-configuration"></a>
### 1.2 Environment Configuration

This step captures the values used throughout the notebook. The prompts use `getpass` so the entries are not echoed back in notebook output.

In [ ]:
# Capture the MCP service URLs used later by the management and endpoint clients.
MCP_MGMT_BASE_URL = getpass("Enter MCP Management base URL: ")
SALES_MCP_BASE_URL = getpass("Enter Sales MCP server base URL: ")
FINANCE_MCP_BASE_URL = getpass("Enter Finance MCP server base URL: ")

# Capture the bearer token used by the MCP management client in this notebook.
AUTH_TOKEN = getpass("Enter MCP Management bearer token: ")

# Capture Teradata database connection details for administrative setup.
TD_DB_HOST = getpass("Enter Teradata database host: ")
TD_ADMIN_USER = getpass("Enter Teradata admin username [dbc]: ") or "dbc"
TD_ADMIN_PASSWORD = getpass("Enter Teradata admin password: ")

# Capture the two endpoint-management users used to compare visibility and access.
ENDPOINT_USER1 = getpass("Enter endpoint management username for user 1 [sales_test_1]: ") or "sales_test_1"
ENDPOINT_PASSWORD1 = getpass(f"Enter password for {ENDPOINT_USER1}: ")
ENDPOINT_USER2 = getpass("Enter endpoint management username for user 2 [sales_test_2]: ") or "sales_test_2"
ENDPOINT_PASSWORD2 = getpass(f"Enter password for {ENDPOINT_USER2}: ")

# Capture server IDs and runtime settings used by MCP management calls.
SALES_SERVER_ID = getpass("Enter Sales MCP server ID [sales-team1]: ") or "sales-team1"
FINANCE_SERVER_ID = getpass("Enter Finance MCP server ID [finance-team1]: ") or "finance-team1"
NODE_POOL = getpass("Enter Kubernetes node pool [default-wl-pool]: ") or "default-wl-pool"
SSL_VERIFY = (getpass("Verify SSL certificates? [y]: ") or "y").lower() != "n"

# Define the demo endpoint paths used by the create, update, and delete examples.
CUSTOM_ENDPOINT_ONE = "/custom1"
CUSTOM_ENDPOINT_TWO = "/custom2"
CUSTOM_ENDPOINT_THREE = "/custom3"


<a id="authentication"></a>
### 1.3 Authentication

The `teradata_agentstack` Access Manager client supports **4 authentication modes** and **3 ways to provide credentials**.

The credential sources are resolved in this order: direct `auth` parameter, environment variables, then YAML config file.

#### Authentication Modes

| Auth Mode | Class | Required Fields |
| --- | --- | --- |
| Bearer Token | `BearerAuth` | `auth_bearer` |
| Basic Auth | `BasicAuth` | `username`, `password` |
| Client Credentials (OAuth2) | `ClientCredentialsAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret` |
| Device Code (OAuth2) | `DeviceCodeAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret`, `auth_device_auth_url` |

This notebook uses `BearerAuth` for MCP management requests.


Create the bearer-auth object used by the MCP management client in this notebook.

In [ ]:
# Create the bearer-auth object used by the MCP management client in this notebook.
bearer_auth = BearerAuth(auth_bearer=AUTH_TOKEN)


<a id="client-initialization"></a>
### 1.4 Client Initialization

This step initializes the MCP management client and the management service wrappers used by the rest of the notebook.

In [ ]:
# Initialize the MCP management client with the prompted bearer token.
mcp_mgmt_client = MCPManagementClient(
    base_url=MCP_MGMT_BASE_URL,
    auth=bearer_auth,
    ssl_verify=SSL_VERIFY,
)

# Create service wrappers for server operations and management health checks.
servers = Servers(client=mcp_mgmt_client)
mgmt_health = MgmtHealth(client=mcp_mgmt_client)


<a id="reusable-helpers"></a>
### 1.5 Reusable Helpers

This step defines the helper used throughout the notebook to display SDK responses consistently.

In [ ]:
# Print SDK responses consistently, whether they are models, dicts, or plain values.
def show_output(label, value):
    print(f'\n{label}:')
    try:
        if hasattr(value, 'model_dump') and callable(value.model_dump):
            pprint(value.model_dump(), sort_dicts=False, width=120)
        elif hasattr(value, 'dict') and callable(value.dict):
            pprint(value.dict(), sort_dicts=False, width=120)
        else:
            pprint(value, sort_dicts=False, width=120)
    except Exception:
        pprint(value, sort_dicts=False, width=120)


<a id="mcp-management"></a>
## 2. MCP Management APIs

Use the MCP management SDK to validate service health, inspect current deployments, create new MCP servers, and confirm state transitions before moving into endpoint-management tasks.

### 2.1 Run Management Health Checks and List Current Deployments

This step confirms the management service is reachable and prints the current MCP server inventory before any deployment or mutation step.


In [ ]:
# Call the management health endpoint to verify the service is reachable.
management_health = mgmt_health.health_check()
show_output("Management health", management_health)


In [ ]:
# List current MCP server deployments before making changes.
current_servers = servers.list()
show_output("Current servers", current_servers)


### 2.2 Deploy the Sales MCP Server

This step builds a parameterized deployment request for the sales MCP server and submits it through the MCP management SDK.


In [ ]:
# Build the sales MCP server deployment request with fixed demo resources and tools.
sales_deploy_request = DeployRequest(
    server_id=SALES_SERVER_ID,
    node_pool=NODE_POOL,
    memory="1Gi",
    cpu="600m",
    initial_replica=1,
    max_replica=1,
    env_vars={
        "TD_DB_HOST": TD_DB_HOST,
        "MCP_TOOLS": "base_readQuery,whoami,dba_tableSpace",
        "RBAC_ENABLED": "true",
        "RBAC_ENDPOINT_ROLE": "TD_ENDPOINT_ADMIN",
        "LOG_LEVEL": "INFO",
    },
)

# Submit the deployment request to the MCP management service.
sales_deploy_result = servers.deploy(body=sales_deploy_request)
show_output("Sales deploy result", sales_deploy_result)


### 2.3 Check Deployment Status

This step fetches the current sales MCP server record directly after deployment so the current state is visible without notebook polling logic.


In [ ]:
# Fetch the sales MCP server record after deployment.
sales_server = servers.get(id=SALES_SERVER_ID)
show_output("Sales server", sales_server)

<a id="endpoint-management"></a>
## 3. Endpoint Management APIs

Use the endpoint-management SDK to prepare user access, initialize per-user clients, validate service readiness, and manage custom endpoint mappings safely.

<a id="endpoint-users"></a>
### 3.1 Create Database Users and Grant Roles Safely

This step creates or resets Teradata users and role grants for endpoint-management validation while keeping the administrative workflow contained in one place.


In [ ]:
# Validate usernames before using them as SQL identifiers.
if not ENDPOINT_USER1.replace("_", "").isalnum() or not ENDPOINT_USER2.replace("_", "").isalnum():
    raise ValueError("Endpoint usernames must contain only letters, numbers, or underscores.")

# Escape single quotes before using passwords inside SQL string literals.
endpoint_password1_sql = ENDPOINT_PASSWORD1.replace("'", "''")
endpoint_password2_sql = ENDPOINT_PASSWORD2.replace("'", "''")

# Connect as the Teradata admin user to create users and assign roles.
with teradatasql.connect(host=TD_DB_HOST, user=TD_ADMIN_USER, password=TD_ADMIN_PASSWORD) as connection:
    with connection.cursor() as cursor:
        # Reset demo users if they already exist.
        cursor.execute(f"DROP USER {ENDPOINT_USER1}", ignoreErrors=[3802])
        cursor.execute(f"DROP USER {ENDPOINT_USER2}", ignoreErrors=[3802])
        # Create endpoint-management users with the prompted passwords.
        cursor.execute(
            f"CREATE USER {ENDPOINT_USER1} AS PERM = 100000000, PASSWORD = '{endpoint_password1_sql}'",
            ignoreErrors=[5612],
        )
        cursor.execute(
            f"CREATE USER {ENDPOINT_USER2} AS PERM = 100000000, PASSWORD = '{endpoint_password2_sql}'",
            ignoreErrors=[5612],
        )
        # Ensure required demo roles exist.
        cursor.execute("CREATE ROLE TD_DATA_ENGINEER", ignoreErrors=[5612])
        cursor.execute("CREATE ROLE TD_DATA_ANALYST", ignoreErrors=[5612])
        cursor.execute("CREATE ROLE TD_ENDPOINT_ADMIN", ignoreErrors=[5612])
        # Grant data and endpoint-admin roles to the demo users.
        cursor.execute(f"GRANT TD_DATA_ENGINEER TO {ENDPOINT_USER1}")
        cursor.execute(f"GRANT TD_DATA_ANALYST TO {ENDPOINT_USER2}")
        cursor.execute(f"GRANT TD_ENDPOINT_ADMIN TO {ENDPOINT_USER1}")
        cursor.execute(f"GRANT TD_ENDPOINT_ADMIN TO {ENDPOINT_USER2}")
        # Read back role assignments so the notebook shows the resulting access setup.
        endpoint_role_rows = cursor.execute(
            f"SELECT Grantee, RoleName FROM dbc.RoleMembersV WHERE Grantee IN ('{ENDPOINT_USER1}', '{ENDPOINT_USER2}') ORDER BY Grantee, RoleName"
        ).fetchall()

# Convert database rows into dictionaries for clearer display.
endpoint_role_assignments = [
    {"grantee": row[0].strip(), "role": row[1].strip()}
    for row in endpoint_role_rows
]

show_output("Endpoint role assignments", endpoint_role_assignments)

<a id="endpoint-clients"></a>
### 3.2 Initialize Per-User Endpoint Management Clients

This step creates one endpoint-management client per user so visibility and authorization behavior can be tested consistently.


In [ ]:
# Initialize an endpoint-management client authenticated as user 1.
ep_client_user1 = MCPEndpointManagementClient(
    base_url=SALES_MCP_BASE_URL,
    auth=BasicAuth(username=ENDPOINT_USER1, password=ENDPOINT_PASSWORD1),
    ssl_verify=SSL_VERIFY,
)
# Initialize a second endpoint-management client authenticated as user 2.
ep_client_user2 = MCPEndpointManagementClient(
    base_url=SALES_MCP_BASE_URL,
    auth=BasicAuth(username=ENDPOINT_USER2, password=ENDPOINT_PASSWORD2),
    ssl_verify=SSL_VERIFY,
)
# Create endpoint service wrappers for each user and a health wrapper for user 1.
endpoints_user1 = Endpoints(client=ep_client_user1)
endpoints_user2 = Endpoints(client=ep_client_user2)
ep_health = EpHealth(client=ep_client_user1)

<a id="endpoint-health-version-visibility"></a>
### 3.3 Run Endpoint Health, Version, and Visibility Checks

This step validates endpoint-management health, retrieves the service version, and compares visible endpoint mappings for each configured user.


In [ ]:
# Call the endpoint-management health endpoint through user 1's client.
endpoint_health = ep_health.health_check()
show_output("Endpoint health", endpoint_health)


In [ ]:
# Retrieve the endpoint-management service version.
endpoint_version = ep_health.version()
show_output("Endpoint version", endpoint_version)


In [ ]:
# List endpoint mappings visible to user 1.
user1_visible_endpoints = endpoints_user1.list()
show_output("User 1 visible endpoints", user1_visible_endpoints)


In [ ]:
# List endpoint mappings visible to user 2.
user2_visible_endpoints = endpoints_user2.list()
show_output("User 2 visible endpoints", user2_visible_endpoints)


<a id="endpoint-create-mappings"></a>
### 3.4 Create Endpoint Mappings

This step creates custom tool endpoints so you can verify endpoint creation and initial visibility through the SDK.


In [ ]:
# Build a primary endpoint mapping for two sales MCP tools.
create_primary_request = EndpointCreateRequest(
    url=CUSTOM_ENDPOINT_ONE,
    resource_type="tools",
    resource_names=["base_readQuery", "dba_tableSpace"],
)

# Create the primary endpoint mapping as user 1.
create_primary_result = endpoints_user1.create(body=create_primary_request)
show_output("Primary endpoint create result", create_primary_result)


In [ ]:
# Build a secondary endpoint mapping for the whoami tool.
create_secondary_request = EndpointCreateRequest(
    url=CUSTOM_ENDPOINT_TWO,
    resource_type="tools",
    resource_names=["whoami"],
)

# Create the secondary endpoint mapping as user 1.
create_secondary_result = endpoints_user1.create(body=create_secondary_request)
show_output("Secondary endpoint create result", create_secondary_result)


<a id="endpoint-update-delete-mappings"></a>
### 3.5 Update and Delete Endpoint Mappings

Create, update, delete, and verify endpoint mappings in separate steps so each operation is easy to run and troubleshoot.


<a id="endpoint-update-mappings"></a>
#### 3.5.1 Update Endpoint Mappings

This step creates and updates endpoint mappings, then confirms the returned SDK state.


In [ ]:
# Build the request for a third endpoint mapping owned by user 2.
create_tertiary_request = EndpointCreateRequest(
    url=CUSTOM_ENDPOINT_THREE,
    resource_type="tools",
    resource_names=["base_readQuery"],
)

# Create the tertiary endpoint mapping.
create_tertiary_result = endpoints_user2.create(body=create_tertiary_request)
show_output("Tertiary endpoint create result", create_tertiary_result)


In [ ]:
# Define the updated tool list for the tertiary endpoint.
update_tertiary_request = EndpointUpdateRequest(
    url=CUSTOM_ENDPOINT_THREE,
    resource_type="tools",
    resource_names=["base_readQuery", "whoami"],
)

# Apply the endpoint update as user 2.
update_tertiary_result = endpoints_user2.update(body=update_tertiary_request)
show_output("Tertiary endpoint update result", update_tertiary_result)


<a id="endpoint-delete-mappings"></a>
#### 3.5.2 Delete Endpoint Mappings

This step deletes endpoint mappings and then verifies the final endpoint visibility for each user.


In [ ]:
# Build a delete request for the primary endpoint mapping.
delete_primary_request = EndpointDeleteRequest(url=CUSTOM_ENDPOINT_ONE, resource_type="tools")

# Delete the primary endpoint mapping as user 1.
delete_primary_result = endpoints_user1.delete(body=delete_primary_request)
show_output("Primary endpoint delete result", delete_primary_result)


In [ ]:
# Build a delete request for the secondary endpoint mapping.
delete_secondary_request = EndpointDeleteRequest(url=CUSTOM_ENDPOINT_TWO, resource_type="tools")

# Delete the secondary endpoint mapping as user 1.
delete_secondary_result = endpoints_user1.delete(body=delete_secondary_request)
show_output("Secondary endpoint delete result", delete_secondary_result)


In [ ]:
# Fetch the final endpoint mappings visible to user 1.
user1_final_list = endpoints_user1.list()
show_output("User 1 final endpoints", user1_final_list)


In [ ]:
# Fetch the final endpoint mappings visible to user 2.
user2_final_list = endpoints_user2.list()
show_output("User 2 final endpoints", user2_final_list)


<a id="additional-operations"></a>
## 4. Additional MCP Server Operations

Use the same workflows to deploy another server, adjust resources, and exercise lifecycle controls to ensure complete server management capability.


### 4.1 Deploy the Finance MCP Server

This step repeats the deployment pattern for a second MCP server and then fetches its current record with a direct SDK call.


In [ ]:
# Build the finance MCP server deployment request with fixed demo resources and tools.
finance_deploy_request = DeployRequest(
    server_id=FINANCE_SERVER_ID,
    node_pool=NODE_POOL,
    memory="2Gi",
    cpu="200m",
    initial_replica=1,
    max_replica=2,
    env_vars={
        "TD_DB_HOST": TD_DB_HOST,
        "MCP_TOOLS": "base_readQuery,whoami",
        "LOG_LEVEL": "INFO",
    },
)

# Submit the finance deployment request to the MCP management service.
finance_deploy_result = servers.deploy(body=finance_deploy_request)
show_output("Finance deploy result", finance_deploy_result)


In [ ]:
# Fetch the finance MCP server record after deployment.
finance_server = servers.get(id=FINANCE_SERVER_ID)
show_output("Finance server", finance_server)


### 4.2 Upgrade Server Resources

This step submits a resource update for the sales MCP server and then rechecks the server inventory to confirm the change.


In [ ]:
# Build the sales MCP server resource upgrade request.
sales_upgrade_request = UpgradeResourcesRequest(
    cpu="700m",
    memory="2Gi",
    initial_replica=1,
    max_replica=1,
)

# Submit the upgrade request for the sales MCP server.
upgrade_result = servers.upgrade(id=SALES_SERVER_ID, body=sales_upgrade_request)
show_output("Upgrade result", upgrade_result)


In [ ]:
# List server inventory after the upgrade request.
server_inventory_after_upgrade = servers.list()
show_output("Server inventory after upgrade", server_inventory_after_upgrade)


<a id="managed-server-lifecycle"></a>
### 4.3 Managed Server Lifecycle Operations

Use these lifecycle cells to stop, start, and restart the managed sales MCP server one operation at a time.


<a id="stop-managed-servers"></a>
#### 4.3.1 Stop Managed Servers

This step stops the managed server and displays its current state immediately after the operation.


In [ ]:
# Stop the sales MCP server.
stop_sales_result = servers.stop(id=SALES_SERVER_ID)
# Refresh the server record after the stop operation.
sales_after_stop = servers.get(id=SALES_SERVER_ID)

# Display the stop response and refreshed server state.
show_output("Stop result", stop_sales_result)
show_output("Sales server after stop", sales_after_stop)


<a id="start-managed-servers"></a>
#### 4.3.2 Start Managed Servers

This step starts the managed server and displays its current state immediately after the operation.


In [ ]:
# Start the sales MCP server.
start_sales_result = servers.start(id=SALES_SERVER_ID)
# Refresh the server record after the start operation.
sales_after_start = servers.get(id=SALES_SERVER_ID)

# Display the start response and refreshed server state.
show_output("Start result", start_sales_result)
show_output("Sales server after start", sales_after_start)


<a id="restart-managed-servers"></a>
#### 4.3.3 Restart Managed Servers

This step restarts the managed server and displays its current state immediately after the operation.


In [ ]:
# Restart the sales MCP server.
restart_sales_result = servers.restart(id=SALES_SERVER_ID)
# Refresh the server record after the restart operation.
sales_after_restart = servers.get(id=SALES_SERVER_ID)


In [ ]:
# Display the restart response and refreshed server state.
show_output("Restart result", restart_sales_result)
show_output("Sales server after restart", sales_after_restart)


<a id="cleanup"></a>
## 5. Cleanup and Results

Undeploy the demo servers when you are finished, then review the outcome and reuse notes for future runs.

### 5.1 Undeploy Servers and Verify Cleanup

This step undeploys the demo MCP servers and prints the final server inventory so cleanup is visible before you finish the notebook.

In [ ]:
# Undeploy the sales MCP server created by the demo.
undeploy_sales_result = servers.undeploy(id=SALES_SERVER_ID)
show_output("Sales undeploy result", undeploy_sales_result)


In [ ]:
# Undeploy the finance MCP server created by the demo.
undeploy_finance_result = servers.undeploy(id=FINANCE_SERVER_ID)
show_output("Finance undeploy result", undeploy_finance_result)


In [ ]:
# List remaining MCP servers after cleanup.
final_server_list = servers.list()
show_output("Final server list", final_server_list)
print("SUCCESS: MCP demo flow completed.")


### 5.2 Results and Interpretation

Run the notebook from top to bottom. Each code cell performs the step for that section directly and shows the returned SDK response so the operator can inspect the current state before moving on.


### 5.3 Summary / Next Steps

This notebook now uses a direct execution pattern: enter the prompted values with `getpass`, run each step in order, and inspect the returned SDK objects after each deploy, endpoint-management, lifecycle, and cleanup action.

**Key points:**
- Inputs are captured with `getpass` at the top of the notebook.
- Each section uses direct MCP or endpoint-management SDK calls without notebook-defined polling loops or wrapper functions.
- Status checks use `servers.get(...)` immediately after the action so the notebook stays simple and readable.
- The final cleanup cell undeploys the demo servers and prints a success message when the flow completes.

**Last validated:** 2026-06-16 (structure and syntax updated for readability; live execution still depends on reachable services, valid credentials, and target infrastructure state).